In [0]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.px_cat_g1v2")

##### Importing from utils and validation_utils package

In [0]:
from utils.custom_utils import Transformations
from validation_utils.test_utils import Validations
tr_obj = Transformations()
va_obj = Validations()

### Silver layer transformations

#### 1. Renaming columns

In [0]:
rename_config = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name)

#### 2. Removing records with no category_id
- This table doesnot have any nulls in category_is 

In [0]:
va_obj.null_check(df = df, primary_col = "category_id")

#### 3. Removing category_id duplicates
- This table doesnot have any category_id duplicates

#### 4. Trimming
- No columns have any leading or trailing spaces

In [0]:
va_obj.whitespace_check(df)

#### 5. Adding ingestion_ts column

In [0]:
df = tr_obj.add_ingestion_timestamp(df)

#### Dataframe sanity check

In [0]:
df.limit(10).display()

#### 6. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.erp_product_category"
if spark.catalog.tableExists(target_table):
    tr_obj.upsert(
        spark = spark,
        df = df,
        key_cols = ["category_id"],
        table = "erp_product_category",
        cdc = "ingestion_ts"
    )
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
SELECT *
FROM lakehouse.silver.erp_product_category
LIMIT 10